# Bedding × Dinomaly — inference tutorial

End-to-end demo of running the 6-channel ALL6 high-res (672×672) Dinomaly pipeline on the bedding val set.

Covers:
1. Loading the saved pipeline (YAML + .pt).
2. Reading a cu3s cube and feeding it to the pipeline.
3. Visualising the VIS / SWIR triplets + score heatmap + GT mask overlay (the 6-channel analog of the lentils notebook's RGB/CIR/custom triptych).
4. Applying the verified-lossless inference speedup recipe (TF32 + bf16 autocast + `torch.compile`) — ~3× faster, AUROC preserved to 5 decimals.

Companion notebooks: `bedding_all6_train_tutorial.ipynb` (build + train) and `bedding_all6_results_tutorial.ipynb` (post-hoc analysis).

In [ ]:
%matplotlib inline
import sys, time
from pathlib import Path

import cuvis
import numpy as np
import torch

# Local helpers (utils.py sits next to this notebook).
sys.path.insert(0, str(Path('.').resolve()))
from utils import (
    resolve_default_config, load_bedding_cu3s_path,
    render_input_triplets, render_inference_panel,
    apply_lossless_speedups,
)

from cuvis_ai_core.pipeline.pipeline import CuvisPipeline
from cuvis_ai_core.utils.node_registry import NodeRegistry
from cuvis_ai_schemas.enums import ExecutionStage
from cuvis_ai_schemas.execution import Context

config = resolve_default_config()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
print('Pipeline YAML:', config['pipeline_yaml'])
print('Pipeline PT  :', config['pipeline_pt'])

## 1 · Load the saved pipeline

We load the high-res 20-epoch ALL6 model. The pipeline graph is: `LentilsAnomalyDataNode` → `MinMaxNormalizer` → `FixedHyperspectralSelector(target_wavelengths=(625,550,450,1450,1200,1050))` → `DinomalyDetector(input_channels=6, image_size=672)` → `decider`.

In [ ]:
registry = NodeRegistry()
registry.load_plugins(str(config['plugins_yaml']))

pipeline = CuvisPipeline.load_pipeline(
    config['pipeline_yaml'],
    weights_path=str(config['pipeline_pt']),
    device=str(device),
    strict_weight_loading=False,
    node_registry=registry,
)
pipeline.torch_layers.eval()
print('Nodes:', [n.name for n in pipeline.graph.nodes])

## 2 · Run inference on a curated val frame

We hand-pick a `_nok_nok_` frame — both sides anomalous — because it gives the clearest qualitative story (multiple foreign objects, easy to see what the model finds). Other interesting frames to try (swap into `FRAME_STEM` below):

- `20250310_153943_frame_121_nok_nok_rdx_rwx` — top TP (score 0.759, multiple PLA pieces + leaf)
- `20250310_084228_frame_30_ok_ok_rd4_rw8` — normal, the `rd4_rw8` sub-recipe that the 672² model correctly rejects
- `20250311_101035_frame_10_nok_ok_rdx_rwx` — edge case: model detects, but GT mask is empty (annotation gap; see `comparisons/ead_baseline/discrepancy_investigation.md`)
- `20250311_101004_frame_9_nok_ok_rdx_rwx` — near-miss FN (just below best-F1 threshold)

In [ ]:
FRAME_STEM = '20250310_153943_frame_121_nok_nok_rdx_rwx'
cu3s_path = load_bedding_cu3s_path(FRAME_STEM, val_root=config['cu3s_val_root'])
assert cu3s_path.is_file(), cu3s_path

# Read the 6-band cube straight from cu3s, mirroring the EfficientAD example's inference pattern.
mesu = cuvis.SessionFile(str(cu3s_path)).get_measurement(0)
cube_hwc = mesu.data['cube'].array.astype(np.float32)
wavelengths_nm = np.asarray(mesu.data['cube'].wavelength, dtype=np.int32)
print(f'cube shape: {cube_hwc.shape}, dtype: {cube_hwc.dtype}, wavelengths: {wavelengths_nm.tolist()}')

In [ ]:
# Visualise the two 3-channel triplets the pipeline sees after the selector reorders into
# (625, 550, 450, 1450, 1200, 1050) nm — VIS-RGB on the left, SWIR-pseudo-RGB on the right.
# Note: the raw cube here is in its native acquisition order; the selector inside the pipeline
# does the reorder. For *display* we replicate that order on the raw cube.

ASC_TO_TARGET_ORDER = []
for nm in (625, 550, 450, 1450, 1200, 1050):
    # Pick the nearest cube band for each target wavelength.
    ASC_TO_TARGET_ORDER.append(int(np.argmin(np.abs(wavelengths_nm - nm))))
cube_ordered = cube_hwc[..., ASC_TO_TARGET_ORDER][None, ...]  # [1, H, W, 6]
fig = render_input_triplets(cube_ordered, title=f'{FRAME_STEM} — VIS (625/550/450 nm) vs SWIR (1450/1200/1050 nm)')
fig.savefig(Path('outputs') / f'{FRAME_STEM}_input_triplets.png', dpi=120, bbox_inches='tight')

In [ ]:
# Build the pipeline input batch. The pipeline expects the raw 6-band cube + wavelengths;
# the selector node reorders to (625,550,450,1450,1200,1050) and MinMaxNormalizer normalizes
# to [0, 1] before the detector. So feed the *raw* cube — not the manually-reordered one.

x_raw = torch.from_numpy(cube_hwc[None, ...]).to(device)             # [1, H, W, 6]
wl   = torch.from_numpy(wavelengths_nm[None, :]).to(device)
batch = {'cube': x_raw, 'wavelengths': wl, 'mesu_index': torch.zeros(1, dtype=torch.int64).to(device),
         'mask': torch.zeros(1, *x_raw.shape[1:3], 1, dtype=torch.bool).to(device)}
ctx = Context(stage=ExecutionStage.INFERENCE, epoch=0, batch_idx=0, global_step=0)

# Time the forward pass.
torch.cuda.synchronize() if device.type == 'cuda' else None
t0 = time.perf_counter()
with torch.inference_mode():
    out = pipeline.forward(batch=batch, context=ctx)
torch.cuda.synchronize() if device.type == 'cuda' else None
dt_ms = (time.perf_counter() - t0) * 1000.0
score_map = out[('dinomaly_detector', 'scores')].detach().float().cpu().numpy()
image_score = float(out[('dinomaly_detector', 'anomaly_score')].item())
print(f'fp32 baseline forward: {dt_ms:.1f} ms/frame   |   image score = {image_score:.4f}')

In [ ]:
# Render the qualitative panel: VIS, SWIR, score heatmap (+ GT contour if available).
MASK_PNG = config['mask_root'] / f'{FRAME_STEM}_mask.png'
gt_mask = None
if MASK_PNG.is_file():
    import cv2
    gt_mask = cv2.imread(str(MASK_PNG), cv2.IMREAD_GRAYSCALE)
    if gt_mask is not None and gt_mask.shape != score_map.shape[1:3]:
        gt_mask = cv2.resize(gt_mask, (score_map.shape[2], score_map.shape[1]),
                             interpolation=cv2.INTER_NEAREST)
fig = render_inference_panel(cube_ordered, score_map, gt_mask=gt_mask,
                              title=f'{FRAME_STEM}  ·  image score = {image_score:.4f}')
fig.savefig(Path('outputs') / f'{FRAME_STEM}_inference_panel.png', dpi=120, bbox_inches='tight')

## 3 · Apply the lossless inference speedup recipe

Three orthogonal optimisations, each verified to preserve metrics on the full val set (pixel AUROC matches fp32 to 5 decimals; image AUROC within ±0.003):

1. `torch.set_float32_matmul_precision('high')` — TF32 matmul on Ampere.
2. `torch.autocast(dtype=torch.bfloat16)` — bf16-mixed for the heavy attention layers.
3. `torch.compile(mode='reduce-overhead')` on the underlying DINOv2 model.

Combined recipe: ~3× faster end-to-end (145 ms → 52 ms/frame on RTX A4000) with no measurable accuracy drift.

In [ ]:
# Apply speedup recipe and measure the new latency.
autocast_ctx = apply_lossless_speedups(pipeline)

# First call after torch.compile is the JIT — discard, then time.
with torch.inference_mode(), autocast_ctx:
    _ = pipeline.forward(batch=batch, context=ctx)
torch.cuda.synchronize() if device.type == 'cuda' else None

samples = []
with torch.inference_mode(), autocast_ctx:
    for _ in range(5):
        if device.type == 'cuda': torch.cuda.synchronize()
        t0 = time.perf_counter()
        out2 = pipeline.forward(batch=batch, context=ctx)
        if device.type == 'cuda': torch.cuda.synchronize()
        samples.append((time.perf_counter() - t0) * 1000.0)
score_map2 = out2[('dinomaly_detector', 'scores')].detach().float().cpu().numpy()
image_score2 = float(out2[('dinomaly_detector', 'anomaly_score')].item())
print(f'after speedup recipe: {np.mean(samples):.1f} ± {np.std(samples):.1f} ms/frame ({len(samples)} timed)')
print(f'image score: fp32={image_score:.5f}  optimized={image_score2:.5f}  (delta={image_score2-image_score:+.5f})')
print(f'score-map correlation vs fp32: {np.corrcoef(score_map.ravel(), score_map2.ravel())[0,1]:.6f}')

## Takeaways

- The 6-channel pipeline reorders the cube to `(625, 550, 450, 1450, 1200, 1050) nm` so the patch-embed-inflation conv sees matched per-slot statistics. We display both triplets (VIS and SWIR) side by side — the bedding analog of the lentils notebook's 3-method triptych.
- The lossless speedup recipe (TF32 + bf16 + `torch.compile`) makes the pipeline ~3× faster with the score map essentially identical (correlation > 0.9997).
- For end-to-end metric verification across the full val set, see
  `examples/bedding_dinomaly/verify_fast_inference_metrics.py` in the cookbook —
  it confirms pixel AUROC, image AUROC, and Dice all stay within ±0.003 of the
  fp32 baseline.
- For the full inference-latency story (Dinomaly vs EfficientAD across resolutions),
  see `comparisons/ead_baseline/ead_multi_resolution_profile.md`.